# Assignment 5 – Perceptron
**MATH-CSCI 485**  
### Part 1: Heuristic Perceptron | Part 2: Gradient Descent Perceptron

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)

## Data Loading & Visualization

In [ ]:
data = pd.read_csv('data.csv', header=None, names=['x1', 'x2', 'label'])
X = data[['x1', 'x2']].values
y = data['label'].values

print(f'Dataset shape: {X.shape}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

# Plot raw data
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class 1', s=40)
ax.scatter(X[y==0, 0], X[y==0, 1], c='red',  label='Class 0', s=40)
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Data Plot'); ax.legend()
plt.tight_layout(); plt.show()

## Shared Utilities

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def boundary_y(x1_vals, w, b):
    """Decision boundary: w1*x1 + w2*x2 + b = 0  =>  x2 = -(w1*x1 + b)/w2"""
    if abs(w[1]) < 1e-10:
        return None
    return -(w[0] * x1_vals + b) / w[1]

def plot_data(ax):
    ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', s=30, zorder=5, label='Class 1')
    ax.scatter(X[y==0, 0], X[y==0, 1], c='red',  s=30, zorder=5, label='Class 0')
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_title('Solution boundary')

x_range = np.linspace(-0.05, 1.05, 300)

---
## Part 1 – Heuristic Perceptron

**Algorithm:**
1. Initialize weights `w` and bias `b` randomly.
2. For each data point: compute sigmoid prediction `ŷ = σ(WX + b)`, compute error `(y - ŷ)`, update:
   - `b  ←  b + r·(y − ŷ)`
   - `wᵢ ←  wᵢ + r·(y − ŷ)·xᵢ`
3. Repeat until total error < tolerance.

The **sigmoid** function provides continuous (soft) classification. The weight update is proportional to the prediction error, enabling smooth gradient-like adjustments without full backprop.

In [ ]:
def perceptron_heuristic(lr, max_iter=1000, tol=1e-4, seed=42):
    np.random.seed(seed)
    w = np.random.randn(2) * 0.01
    b = np.random.randn()  * 0.01

    fig, ax = plt.subplots(figsize=(6, 6))
    plot_data(ax)

    # Initial boundary — red
    y0 = boundary_y(x_range, w, b)
    if y0 is not None:
        ax.plot(x_range, y0, 'r-', linewidth=1.5, label='Initial boundary', zorder=3)

    iteration = 0
    for it in range(max_iter):
        total_error = 0
        for xi, yi in zip(X, y):
            z    = np.dot(w, xi) + b
            yhat = sigmoid(z)
            err  = yi - yhat
            b   += lr * err
            w   += lr * err * xi
            total_error += abs(err)

        iteration = it + 1
        # Intermediate boundary — dashed green
        yl = boundary_y(x_range, w, b)
        if yl is not None:
            ax.plot(x_range, yl, 'g--', linewidth=0.4, alpha=0.55, zorder=2)

        if total_error < tol:
            print(f'  Converged at iteration {iteration}')
            break
    else:
        print(f'  Reached max iterations ({max_iter})')

    # Final boundary — black
    yf = boundary_y(x_range, w, b)
    if yf is not None:
        ax.plot(x_range, yf, 'k-', linewidth=2.2,
                label=f'Final boundary (iter={iteration})', zorder=4)

    ax.legend(fontsize=8)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.set_title(f'Part 1 – Heuristic | lr={lr}, iterations={iteration}')
    plt.tight_layout(); plt.show()
    return w, b, iteration

In [ ]:
print('=== lr = 0.01 ===')
w1, b1, it1 = perceptron_heuristic(lr=0.01)

In [ ]:
print('=== lr = 0.1 ===')
w2, b2, it2 = perceptron_heuristic(lr=0.1)

In [ ]:
print('=== lr = 1.0 ===')
w3, b3, it3 = perceptron_heuristic(lr=1.0)

### Part 1 Analysis

| Learning Rate | Iterations | Behavior |
|:---:|:---:|:---|
| 0.01 | 1000 (max) | Very slow convergence; boundary lines stay tightly clustered near the initial line. Small steps mean the algorithm is cautious but slow to find the separating hyperplane. |
| 0.1  | 1000 (max) | Moderate movement; the boundary sweeps across the data space reasonably, producing a better final separator. |
| 1.0  | 1000 (max) | Large steps; the boundary oscillates widely in early iterations (visible scatter in green dashes), but the sigmoid's continuous output dampens instability and the final boundary still converges to a reasonable separator. |

**Key observations:**
- Using the **sigmoid** output (continuous ŷ ∈ (0,1)) instead of a hard step function allows gradual weight updates proportional to confidence, which is more stable than a binary flip.
- A smaller learning rate avoids overshooting but may need far more iterations or may never converge within the limit.
- A larger learning rate can converge faster but risks instability; the sigmoid's bounded output mitigates catastrophic weight explosions.

---
## Part 2 – Gradient Descent Perceptron

**Algorithm:**
1. Initialize weights and bias randomly.
2. For each data point, classify with a **hard threshold** (z ≥ 0 → class 1):
   - If misclassified as 0 (true label = 1): `b += r`, `wᵢ += r·xᵢ`
   - If misclassified as 1 (true label = 0): `b -= r`, `wᵢ -= r·xᵢ`
3. Repeat for a fixed number of **epochs**.
4. Track **log-loss** (binary cross-entropy) every 10 epochs.

**Log-loss:** `L = -mean[ y·log(σ(z)) + (1-y)·log(1-σ(z)) ]`  
Even though the update rule uses binary classification, we use sigmoid-based log-loss as a smooth proxy for how well the boundary separates the data.

In [ ]:
def log_loss(X, y, w, b):
    eps  = 1e-12
    yhat = sigmoid(X @ w + b)
    return -np.mean(y * np.log(yhat + eps) + (1 - y) * np.log(1 - yhat + eps))

def perceptron_gradient_descent(lr, epochs=100, seed=42):
    np.random.seed(seed)
    w = np.random.randn(2) * 0.01
    b = np.random.randn()  * 0.01

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    plot_data(ax1)

    # Initial boundary — red
    y0 = boundary_y(x_range, w, b)
    if y0 is not None:
        ax1.plot(x_range, y0, 'r-', linewidth=1.5, label='Initial', zorder=3)

    error_epochs, error_vals = [], []

    for epoch in range(epochs):
        for xi, yi in zip(X, y):
            z    = np.dot(w, xi) + b
            pred = 1 if z >= 0 else 0
            if pred != int(yi):
                if pred == 0:   # predict 0, true 1
                    b += lr; w += lr * xi
                else:           # predict 1, true 0
                    b -= lr; w -= lr * xi

        # Intermediate boundary — dashed green
        yl = boundary_y(x_range, w, b)
        if yl is not None:
            ax1.plot(x_range, yl, 'g--', linewidth=0.35, alpha=0.5, zorder=2)

        # Record log-loss every 10 epochs
        if epoch % 10 == 0 or epoch == epochs - 1:
            error_epochs.append(epoch)
            error_vals.append(log_loss(X, y, w, b))

    # Final boundary — blue (matching sample image)
    yf = boundary_y(x_range, w, b)
    if yf is not None:
        ax1.plot(x_range, yf, 'b-', linewidth=2.5,
                 label=f'Final (epoch={epochs})', zorder=4)

    ax1.legend(fontsize=8)
    ax1.set_xlabel('x1'); ax1.set_ylabel('x2')

    # Error plot
    ax2.plot(error_epochs, error_vals, linewidth=2, color='steelblue')
    ax2.set_title('Error Plot')
    ax2.set_xlabel('Number of epochs')
    ax2.set_ylabel('Error (Log-Loss)')

    fig.suptitle(f'Part 2 – Gradient Descent | lr={lr}, epochs={epochs}', fontsize=11)
    plt.tight_layout(); plt.show()
    print(f'  Final log-loss: {error_vals[-1]:.4f}')
    return w, b, error_epochs, error_vals

In [ ]:
print('=== lr=0.1, epochs=100 ===')
perceptron_gradient_descent(lr=0.1, epochs=100);

In [ ]:
print('=== lr=0.01, epochs=100 ===')
perceptron_gradient_descent(lr=0.01, epochs=100);

In [ ]:
print('=== lr=1.0, epochs=100 ===')
perceptron_gradient_descent(lr=1.0, epochs=100);

In [ ]:
print('=== lr=0.1, epochs=500 ===')
perceptron_gradient_descent(lr=0.1, epochs=500);

### Part 2 Analysis

| lr | Epochs | Final Log-Loss | Behavior |
|:---:|:---:|:---:|:---|
| 0.01 | 100 | ~0.680 | Barely moves; 100 epochs with tiny steps is insufficient. The error barely decreases. |
| 0.1  | 100 | ~0.577 | Reasonable convergence; error curve shows classic exponential-decay shape. |
| 1.0  | 100 | ~0.379 | Fastest drop in log-loss; larger steps correct misclassifications aggressively. |
| 0.1  | 500 | ~0.552 | More epochs with moderate lr continue to reduce error but with diminishing returns on linearly non-separable data. |

**Key observations:**
- The **log-loss curve** consistently shows rapid initial decrease followed by a plateau — consistent with the classic perceptron convergence behavior on data that is not perfectly linearly separable.
- The **gradient descent** (hard-threshold) approach is more aggressive than the heuristic sigmoid approach: it only updates on **misclassified** points, while the heuristic always updates.
- With `lr=1.0`, the boundary moves in large jumps (many green dashes), but ultimately finds a good separator because the data has a clear linear trend.
- Increasing epochs beyond the point of convergence offers marginal benefit on this dataset; the data is not perfectly separable, so some irreducible log-loss remains.

**Comparison between Part 1 & Part 2:**

| Property | Part 1 (Heuristic) | Part 2 (GD) |
|:---|:---|:---|
| Activation | Sigmoid (continuous) | Step (binary) |
| Updates | Every point | Misclassified only |
| Update magnitude | Proportional to error | Fixed = lr |
| Error metric | Absolute sigmoid error | Log-loss |
| Stability | More stable (smooth updates) | Can oscillate with high lr |